In [2]:
import os
import shutil
import random

# ---------------------------
# SETTINGS
# ---------------------------
SOURCE_DIR = r"C:\project\dataformob"  # original data
DEST_DIR   = "datamob"                # output folder

TRAIN_RATIO = 0.7
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

random.seed(42)

# ---------------------------
# CREATE FOLDERS
# ---------------------------
for split in ["train", "val", "test"]:
    for cls in os.listdir(SOURCE_DIR):
        os.makedirs(os.path.join(DEST_DIR, split, cls), exist_ok=True)

# ---------------------------
# SPLITTING
# ---------------------------
for cls in os.listdir(SOURCE_DIR):
    files = os.listdir(os.path.join(SOURCE_DIR, cls))
    random.shuffle(files)

    total = len(files)
    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)

    splits = {
        "train": files[:train_end],
        "val": files[train_end:val_end],
        "test": files[val_end:]
    }

    for split, split_files in splits.items():
        for file in split_files:
            src_path = os.path.join(SOURCE_DIR, cls, file)
            dst_path = os.path.join(DEST_DIR, split, cls, file)
            shutil.copy(src_path, dst_path)

    print(f"✅ {cls}: {len(splits['train'])} train, {len(splits['val'])} val, {len(splits['test'])} test")

print("\n✅ DONE: Dataset split into train/val/test")


✅ others: 778 train, 166 val, 168 test
✅ papaya: 774 train, 165 val, 167 test
✅ pepper: 795 train, 170 val, 171 test

✅ DONE: Dataset split into train/val/test


In [13]:
for split in ["train","val","test"]:
    print(f"\n{split.upper()}:")
    for cls in os.listdir(os.path.join("datamob", split)):
        count = len(os.listdir(os.path.join("datamob", split, cls)))
        print(cls, "→", count)



TRAIN:
others → 778
papaya → 772
pepper → 795

VAL:
others → 166
papaya → 164
pepper → 170

TEST:
others → 168
papaya → 166
pepper → 170


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image, UnidentifiedImageError
import os

# ----------------------------
# CONFIG
# ----------------------------
DATA_DIR = r"C:\project\datamob"
BATCH = 32
EPOCHS = 15
LR = 0.0003
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

# ----------------------------
# SAFE IMAGE LOADER (SKIPS CORRUPTED IMAGES)
# ----------------------------
def safe_loader(path):
    try:
        img = Image.open(path).convert("RGB")
        return img
    except UnidentifiedImageError:
        print("❌ Corrupted image skipped:", path)
        return None

# ----------------------------
# CLEAN DATASET CLASS (REMOVES BAD IMAGES)
# ----------------------------
from torchvision.datasets import ImageFolder

class CleanImageFolder(ImageFolder):
    def __getitem__(self, index):
        while True:
            path, label = self.samples[index]
            img = self.loader(path)

            # Skip corrupted images
            if img is None:
                print("⚠ Removing corrupted:", path)
                self.samples.pop(index)
                if len(self.samples) == 0:
                    raise RuntimeError("All images are corrupted!")
                index = index % len(self.samples)
                continue

            if self.transform:
                img = self.transform(img)
            return img, label

# ----------------------------
# TRANSFORMS
# ----------------------------
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# ----------------------------
# DATASETS (NOW USING CLEANIMAGEFOLDER)
# ----------------------------
train_set = CleanImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tf, loader=safe_loader)
val_set   = CleanImageFolder(os.path.join(DATA_DIR, "val"),   transform=val_tf, loader=safe_loader)
test_set  = CleanImageFolder(os.path.join(DATA_DIR, "test"),  transform=val_tf, loader=safe_loader)

train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=BATCH)
test_loader  = DataLoader(test_set, batch_size=BATCH)

CLASSES = train_set.classes
print("Classes:", CLASSES)

# ----------------------------
# MODEL
# ----------------------------
model = models.mobilenet_v2(weights="IMAGENET1K_V1")
model.classifier[1] = nn.Linear(model.last_channel, len(CLASSES))
model = model.to(DEVICE)

# ----------------------------
# LOSS & OPTIMIZER
# ----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ----------------------------
# TRAIN LOOP
# ----------------------------
best_acc = 0

for epoch in range(EPOCHS):
    model.train()
    train_correct = 0
    train_total = 0
    train_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        preds = model(imgs)
        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += (preds.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_acc = 100 * train_correct / train_total

    # ----------------------------
    # VALIDATION
    # ----------------------------
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")

    # SAVE BEST MODEL
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "mobilenet_seed.pth")
        print("✅ Model saved")

# ----------------------------
# TEST EVALUATION
# ----------------------------
print("\nTesting best model...")
model.load_state_dict(torch.load("mobilenet_seed.pth"))
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

print(f"\n✅ FINAL TEST ACCURACY = {100 * test_correct / test_total:.2f}%")


Using device: cpu
Classes: ['others', 'papaya', 'pepper']
Epoch 1/15 | Train: 94.67% | Val: 98.40%
✅ Model saved
Epoch 2/15 | Train: 98.76% | Val: 98.80%
✅ Model saved
Epoch 3/15 | Train: 99.62% | Val: 99.00%
✅ Model saved
Epoch 4/15 | Train: 99.36% | Val: 98.80%
Epoch 5/15 | Train: 98.64% | Val: 98.60%
Epoch 6/15 | Train: 98.34% | Val: 98.60%
Epoch 7/15 | Train: 99.91% | Val: 99.40%
✅ Model saved
Epoch 8/15 | Train: 99.74% | Val: 99.00%
Epoch 9/15 | Train: 97.31% | Val: 98.80%
Epoch 10/15 | Train: 99.36% | Val: 99.00%
Epoch 11/15 | Train: 99.96% | Val: 99.00%
Epoch 12/15 | Train: 100.00% | Val: 99.00%
Epoch 13/15 | Train: 99.91% | Val: 98.60%
Epoch 14/15 | Train: 99.62% | Val: 99.00%
Epoch 15/15 | Train: 99.91% | Val: 99.00%

Testing best model...

✅ FINAL TEST ACCURACY = 99.80%


In [11]:
from PIL import Image, UnidentifiedImageError
import os

ROOT = r"C:\project\datamob"

bad_files = []

for folder, _, files in os.walk(ROOT):
    for file in files:
        path = os.path.join(folder, file)
        try:
            Image.open(path).verify()  # test file integrity
        except:
            print("❌ BAD FILE REMOVED:", path)
            bad_files.append(path)

for f in bad_files:
    os.remove(f)

print("✅ Cleaning complete. Total removed:", len(bad_files))


❌ BAD FILE REMOVED: C:\project\datamob\test\papaya\20251120_100107(0).jpg
❌ BAD FILE REMOVED: C:\project\datamob\test\pepper\pepper 844.jpg
❌ BAD FILE REMOVED: C:\project\datamob\val\papaya\20251120_100104(0).jpg
✅ Cleaning complete. Total removed: 3
